# Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
from IPython.display import display

# Configurações

In [2]:
DATA_DIR = Path("../data")

PATH_GEOINFO = DATA_DIR / "processed/geoinfo_artigos_processado.csv"
PATH_GOOGLE = DATA_DIR / "processed/citacoes_geoinfo.csv"
PATH_OPENALEX = DATA_DIR / "processed/citacoes_geoinfo_metadados_preenchidos.csv"

OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
COLUNAS_GEOINFO = [
    "titulo",
    "ano",
    "autores",
    "instituicoes",
    "edicao",
    "identificador",
    "url_edicao",
    "url_artigo",
    "url_metadata",
    "numero_edicao"
]

COLUNAS_CITANTES = [
    "titulo_original",
    "ano_original",
    "titulo",
    "ano",
    "autores",
    "instituicoes",
    "idioma",
    "pais",
    "veiculo_publicacao",
    "doi",
    "url"
]

COLUNAS_OPENALEX = [
    *COLUNAS_CITANTES,
    "tipo_documento",
    "topico",
    "subcampo",
    "campo",
    "dominio_tematico",
    "fonte_publicacao",
    "status_acesso_aberto",
    "openalex_id"
]

COLUNAS_TEXTO = [
    "autores",
    "instituicoes",
    "idioma",
    "pais",
    "veiculo_publicacao",
    "tipo_documento",
    "topico",
    "subcampo",
    "campo",
    "dominio_tematico",
    "fonte_publicacao"
]

COLUNAS_CONSOLIDADAS = [
    "id_geoinfo",
    "id_citante",
    "titulo",
    "titulo_padronizado",
    "ano",
    "autores",
    "instituicoes",
    "idioma_padronizado",
    "pais_padronizado",
    "veiculo_publicacao",
    "doi_normalizado",
    "url",
    "tipo_documento",
    "topico",
    "subcampo",
    "campo",
    "dominio_tematico",
    "fonte_publicacao",
    "status_acesso_aberto",
    "openalex_id"
]

# Carregamento dos dados

In [4]:
df_geoinfo = pd.read_csv(PATH_GEOINFO, encoding="utf-8")
df_google = pd.read_csv(PATH_GOOGLE, encoding="utf-8")
df_openalex = pd.read_csv(PATH_OPENALEX, encoding="utf-8")

In [5]:
print(f"GEOINFO:       {df_geoinfo.shape[0]:,} registros")
print(f"Google Scholar:{df_google.shape[0]:,} registros")
print(f"OpenAlex:      {df_openalex.shape[0]:,} registros")

GEOINFO:       682 registros
Google Scholar:2,999 registros
OpenAlex:      2,999 registros


In [6]:
df_geoinfo.head(3)

,titulo,ano,autores,instituicoes,edicao,identificador,url_edicao,url_artigo,url_metadata,numero_edicao
0,Reproducible and empirical method refines Frac...,2025,"1 Adorno, Bruno Vargas 2 Nesbitt, Lorien 3 Ama...",1 National Institute for Space Research (INPE)...,25a. Edição São José dos Campos 2025,8JMKD2USPTW34P/4DKBAQH,http://urlib.net/ibi/8JMKD2USPTW34P/4DKC4FB,http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mt...,http://mtc-m16c.sid.inpe.br/sid.inpe.br/mtc-m1...,25
1,Potential Effects of Legal Reserve Exclusion o...,2025,"1 Andrade, Pedro Ribeiro 2 Rodrigues, Erick Te...",1 National Institute for Space Research (INPE)...,25a. Edição São José dos Campos 2025,8JMKD2USPTW34P/4DKBE9S,http://urlib.net/ibi/8JMKD2USPTW34P/4DKC4FB,http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mt...,http://mtc-m16c.sid.inpe.br/sid.inpe.br/mtc-m1...,25
2,Bill 191/2020: Illegal Mining and Land Use Lan...,2025,"1 Chuizaca-Espinoza, Isabel Adriana 2 Amaral, ...",1 National Institute for Space Research (INPE)...,25a. Edição São José dos Campos 2025,8JMKD2USPTW34P/4DKBSAB,http://urlib.net/ibi/8JMKD2USPTW34P/4DKC4FB,http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mt...,http://mtc-m16c.sid.inpe.br/sid.inpe.br/mtc-m1...,25


In [7]:
df_google.head(3)

,titulo_original,ano_original,titulo,ano,autores,instituicoes,idioma,pais,veiculo_publicacao,doi,url
0,Implementing a new automatic deforestation mo...,2025,Beyond the reporting of disturbed areas: the u...,2026.0,"W Leal Filho, FC Alves, MS Reis, VL Camilotti…...",NaN,NaN,NaN,Springer,NaN,https://link.springer.com/article/10.1186/s405...
1,Implementing a new automatic deforestation mon...,2025,Configuration Assessment of Deter-RT: a New SA...,2025.0,"MS Reis, J Doblas, LH Gusmão… - Rev. Bras …, 2025",NaN,NaN,NaN,researchgate.net,NaN,https://www.researchgate.net/profile/Mariane-R...
2,Retrieval of land-use history in shifting cult...,2025,Revelando Dinâmicas da Agricultura Itinerante ...,2025.0,"MS Reis, ÉT Rodrigues, E Gomes… - Rev. Bras. C...",NaN,NaN,NaN,seer.ufu.br,NaN,https://seer.ufu.br/index.php/revistabrasileir...


In [8]:
df_openalex.head(3)

,titulo_original,ano_original,titulo,ano,autores,instituicoes,idioma,pais,veiculo_publicacao,doi,url,tipo_documento,topico,subcampo,campo,dominio_tematico,fonte_publicacao,status_acesso_aberto,openalex_id
0,Implementing a new automatic deforestation mo...,2025,Beyond the reporting of disturbed areas: the u...,2026.0,"W Leal Filho, FC Alves, MS Reis, VL Camilotti…...",Instituto Nacional de Pesquisas Espaciais; Uni...,en,BR,Springer,https://doi.org/10.1186/s40562-025-00448-9,https://link.springer.com/article/10.1186/s405...,article,"Conservation, Biodiversity, and Resource Manag...",Global and Planetary Change,Environmental Science,Physical Sciences,Geoscience Letters,gold,https://openalex.org/W7119509551
1,Implementing a new automatic deforestation mon...,2025,Configuration Assessment of Deter-RT: a New SA...,2025.0,"MS Reis, J Doblas, LH Gusmão… - Rev. Bras …, 2025",NaN,en,NaN,researchgate.net,NaN,https://www.researchgate.net/profile/Mariane-R...,article,Remote Sensing and LiDAR Applications,Environmental Engineering,Environmental Science,Physical Sciences,Biblioteca Digital da Memória Científica do IN...,green,https://openalex.org/W7124672439
2,Retrieval of land-use history in shifting cult...,2025,Revelando Dinâmicas da Agricultura Itinerante ...,2025.0,"MS Reis, ÉT Rodrigues, E Gomes… - Rev. Bras. C...",SEM_MATCH,NaN,NaN,seer.ufu.br,NaN,https://seer.ufu.br/index.php/revistabrasileir...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print("Colunas GEOINFO:")
print(df_geoinfo.columns.tolist())

print("\nColunas Google Scholar:")
print(df_google.columns.tolist())

print("\nColunas OpenAlex:")
print(df_openalex.columns.tolist())

Colunas GEOINFO:
['titulo', 'ano', 'autores', 'instituicoes', 'edicao', 'identificador', 'url_edicao', 'url_artigo', 'url_metadata', 'numero_edicao']

Colunas Google Scholar:
['titulo_original', 'ano_original', 'titulo', 'ano', 'autores', 'instituicoes', 'idioma', 'pais', 'veiculo_publicacao', 'doi', 'url']

Colunas OpenAlex:
['titulo_original', 'ano_original', 'titulo', 'ano', 'autores', 'instituicoes', 'idioma', 'pais', 'veiculo_publicacao', 'doi', 'url', 'tipo_documento', 'topico', 'subcampo', 'campo', 'dominio_tematico', 'fonte_publicacao', 'status_acesso_aberto', 'openalex_id']


In [10]:
def verificar_colunas(df, esperadas, nome):
    faltantes = set(esperadas) - set(df.columns)

    if faltantes:
        print(f"{nome}: colunas ausentes -> {faltantes}")
    else:
        print(f"{nome}: estrutura OK")


verificar_colunas(df_geoinfo, COLUNAS_GEOINFO, "GEOINFO")
verificar_colunas(df_google, COLUNAS_CITANTES, "Google Scholar")
verificar_colunas(df_openalex, COLUNAS_OPENALEX, "OpenAlex")

GEOINFO: estrutura OK
Google Scholar: estrutura OK
OpenAlex: estrutura OK


In [11]:
def resumo_nulos(df, nome):
    resultado = (df.isna().sum().to_frame("nulos"))
    resultado["percentual"] = (resultado["nulos"] / len(df) * 100).round(2)

    print(f"\n{nome}")
    display(resultado.sort_values("nulos", ascending=False))

resumo_nulos(df_geoinfo, "GEOINFO")
resumo_nulos(df_google, "Google Scholar")
resumo_nulos(df_openalex, "OpenAlex")


GEOINFO


,nulos,percentual
titulo,0,0.0
ano,0,0.0
autores,0,0.0
instituicoes,0,0.0
edicao,0,0.0
identificador,0,0.0
url_edicao,0,0.0
url_artigo,0,0.0
url_metadata,0,0.0
numero_edicao,0,0.0



Google Scholar


,nulos,percentual
doi,2999,100.00
instituicoes,2999,100.00
idioma,2999,100.00
pais,2999,100.00
ano,418,13.94
url,330,11.00
veiculo_publicacao,213,7.10
titulo,0,0.00
ano_original,0,0.00
titulo_original,0,0.00



OpenAlex


,nulos,percentual
fonte_publicacao,2962,98.77
pais,2959,98.67
idioma,2957,98.60
doi,2956,98.57
subcampo,2951,98.40
topico,2951,98.40
campo,2951,98.40
dominio_tematico,2951,98.40
status_acesso_aberto,2950,98.37
openalex_id,2950,98.37


# Funções auxiliares

In [12]:
def normalizar_texto(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()
    valor = re.sub(r"\s+", " ", valor)

    return valor

In [ ]:
def normalizar_titulo(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()

    # normaliza caracteres Unicode
    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(c for c in valor if not unicodedata.combining(c))

    # remove pontuação
    valor = re.sub(r"[^\w\s]", " ", valor)

    # normaliza espaços
    valor = re.sub(r"\s+", " ", valor).strip()

    return valor

# Tratamento e Padronização

In [14]:
for df in [df_geoinfo, df_google, df_openalex]:
    for coluna in df.columns:
        if df[coluna].dtype == "object":
            df[coluna] = df[coluna].apply(normalizar_texto)

## Padronização dos títulos

In [15]:
df_google["titulo"] = (df_google["titulo"].apply(normalizar_titulo))
df_openalex["titulo"] = (df_openalex["titulo"].apply(normalizar_titulo))
df_geoinfo["titulo"] = (df_geoinfo["titulo"].apply(normalizar_titulo))

In [16]:
df_google["titulo_original"] = (df_google["titulo_original"].apply(normalizar_titulo))
df_openalex["titulo_original"] = (df_openalex["titulo_original"].apply(normalizar_titulo))

## Padronizar anos

In [17]:
for df in [df_geoinfo, df_google, df_openalex]:
    for coluna in ["ano", "ano_original"]:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype("Int64")

In [18]:
for df in [df_geoinfo, df_google, df_openalex]:
    for coluna in df.columns:
        if df[coluna].dtype == "object":
            df[coluna] = df[coluna].apply(normalizar_texto)

## Padronização dos campos textuais

In [19]:
for df in [df_google, df_openalex]:
    for coluna in COLUNAS_TEXTO:
        if coluna in df.columns:
            df[coluna] = df[coluna].apply(normalizar_texto)

## Padronização do DOI

In [ ]:
def normalizar_doi(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()
    valor = re.sub(r"^https?://doi\.org/", "", valor)
    valor = re.sub(r"^doi:\s*", "", valor)

    return valor.strip()

In [ ]:
df_google["doi"] = (df_google["doi"].apply(normalizar_doi))
df_openalex["doi"] = (df_openalex["doi"].apply(normalizar_doi))

# Integração

In [20]:
df_geoinfo["chave_geoinfo"] = (df_geoinfo["titulo"] + "|" + df_geoinfo["ano"].astype("string"))

In [21]:
df_google["chave_geoinfo"] = (df_google["titulo_original"] + "|" + df_google["ano_original"].astype("string"))
df_openalex["chave_geoinfo"] = (df_openalex["titulo_original"] + "|" + df_openalex["ano_original"].astype("string"))

In [22]:
# verificacao
chaves_geoinfo = set(df_geoinfo["chave_geoinfo"].dropna())

df_google["geoinfo_encontrado"] = (df_google["chave_geoinfo"].isin(chaves_geoinfo))
df_openalex["geoinfo_encontrado"] = (df_openalex["chave_geoinfo"].isin(chaves_geoinfo))

In [23]:
print("Google Scholar:")
print(df_google["geoinfo_encontrado"].value_counts())

print("\nOpenAlex:")
print(df_openalex["geoinfo_encontrado"].value_counts())

Google Scholar:
geoinfo_encontrado
True    2999
Name: count, dtype: int64

OpenAlex:
geoinfo_encontrado
True    2999
Name: count, dtype: int64


In [24]:
df_google[~df_google["geoinfo_encontrado"]][["titulo_original", "ano_original"]].drop_duplicates().head(20)

,titulo_original,ano_original


In [25]:
mapa_geoinfo = (df_geoinfo.drop_duplicates("chave_geoinfo").set_index("chave_geoinfo")["identificador"])

df_google["id_geoinfo"] = (df_google["chave_geoinfo"].map(mapa_geoinfo))
df_openalex["id_geoinfo"] = (df_openalex["chave_geoinfo"].map(mapa_geoinfo))

In [26]:
# identificador trab citante
df_google["chave_citante"] = (df_google["titulo"] + "|" + df_google["ano"].astype("string"))

df_openalex["chave_citante"] = (df_openalex["titulo"] + "|" + df_openalex["ano"].astype("string"))

In [ ]:
def criar_chave_citante(df):
    return (df["doi"].fillna(df["chave_citante"]))

df_google["id_citante"] = criar_chave_citante(df_google)
df_openalex["id_citante"] = criar_chave_citante(df_openalex)

# Autores

In [ ]:
def extrair_autores_geoinfo(valor):
    if pd.isna(valor):
        return []

    texto = str(valor)
    padrao = r"(\d+)\s+(.+?)(?=\s+\d+\s+|$)"

    resultados = re.findall(padrao, texto)

    return [
        {
            "ordem": int(numero),
            "autor_original": nome.strip()
        }
        for numero, nome in resultados
    ]

In [ ]:
def padronizar_nome_autor_geoinfo(nome):
    if pd.isna(nome):
        return np.nan

    nome = str(nome).strip()

    if "," not in nome:
        return nome

    sobrenome, nomes = nome.split(",", 1)
    return f"{nomes.strip()} {sobrenome.strip()}"

In [ ]:
autores_registros = []

for _, row in df_geoinfo.iterrows():
    autores = extrair_autores_geoinfo(row["autores"])

    for autor in autores:
        autores_registros.append({
            "id_geoinfo": row["identificador"],
            "ordem": autor["ordem"],
            "autor_original": autor["autor_original"],
            "autor_padronizado": padronizar_nome_autor_geoinfo(autor["autor_original"])
        })

In [ ]:
df_autores_geoinfo = pd.DataFrame(autores_registros)
display(df_autores_geoinfo.head(10))

,id_geoinfo,ordem,autor_original,autor_padronizado
0,8JMKD2USPTW34P/4DKBAQH,1,"Adorno, Bruno Vargas",Bruno Vargas Adorno
1,8JMKD2USPTW34P/4DKBAQH,2,"Nesbitt, Lorien",Lorien Nesbitt
2,8JMKD2USPTW34P/4DKBAQH,3,"Amaral, Silvana",Silvana Amaral
3,8JMKD2USPTW34P/4DKBE9S,1,"Andrade, Pedro Ribeiro",Pedro Ribeiro Andrade
4,8JMKD2USPTW34P/4DKBE9S,2,"Rodrigues, Erick Teixeira",Erick Teixeira Rodrigues
5,8JMKD2USPTW34P/4DKBE9S,3,"Simoes, Rolf E. O.",Rolf E. O. Simoes
6,8JMKD2USPTW34P/4DKBE9S,4,"Escada, Maria Isabel Sobral",Maria Isabel Sobral Escada
7,8JMKD2USPTW34P/4DKBSAB,1,"Chuizaca-Espinoza, Isabel Adriana",Isabel Adriana Chuizaca-Espinoza
8,8JMKD2USPTW34P/4DKBSAB,2,"Amaral, Silvana",Silvana Amaral
9,8JMKD2USPTW34P/4DKBDA8,1,"Costa, Gabriel F.",Gabriel F. Costa


# Instituições

-> isso aqui tem tratar melhor

In [ ]:
def extrair_instituicoes_geoinfo(valor):
    if pd.isna(valor):
        return []

    texto = str(valor)
    padrao = r"(\d+)\s+(.+?)(?=\s+\d+\s+|$)"

    resultados = re.findall(padrao, texto)

    return [
        {
            "ordem": int(numero),
            "instituicao_original": nome.strip()
        }
        for numero, nome in resultados
    ]

In [ ]:
instituicoes_registros = []

for _, row in df_geoinfo.iterrows():
    autores = extrair_autores_geoinfo(row["autores"])
    instituicoes = extrair_instituicoes_geoinfo(row["instituicoes"])
    mapa_instituicoes = {item["ordem"]: item["instituicao_original"] for item in instituicoes}

    for autor in autores:
        instituicoes_registros.append({
            "id_geoinfo": row["identificador"],
            "ordem": autor["ordem"],
            "autor_original": autor["autor_original"],
            "autor_padronizado": padronizar_nome_autor_geoinfo(autor["autor_original"]),
            "instituicao_original": mapa_instituicoes.get(autor["ordem"])
        })

In [ ]:
df_autores_instituicoes_geoinfo = pd.DataFrame(instituicoes_registros)

In [ ]:
display(df_autores_instituicoes_geoinfo.head(10))

,id_geoinfo,ordem,autor_original,autor_padronizado,instituicao_original
0,8JMKD2USPTW34P/4DKBAQH,1,"Adorno, Bruno Vargas",Bruno Vargas Adorno,National Institute for Space Research (INPE)
1,8JMKD2USPTW34P/4DKBAQH,2,"Nesbitt, Lorien",Lorien Nesbitt,University of British Columbia
2,8JMKD2USPTW34P/4DKBAQH,3,"Amaral, Silvana",Silvana Amaral,National Institute for Space Research (INPE)
3,8JMKD2USPTW34P/4DKBE9S,1,"Andrade, Pedro Ribeiro",Pedro Ribeiro Andrade,National Institute for Space Research (INPE)
4,8JMKD2USPTW34P/4DKBE9S,2,"Rodrigues, Erick Teixeira",Erick Teixeira Rodrigues,National Institute for Space Research (INPE)
5,8JMKD2USPTW34P/4DKBE9S,3,"Simoes, Rolf E. O.",Rolf E. O. Simoes,Open Geo Hub (OGH)
6,8JMKD2USPTW34P/4DKBE9S,4,"Escada, Maria Isabel Sobral",Maria Isabel Sobral Escada,National Institute for Space Research (INPE)
7,8JMKD2USPTW34P/4DKBSAB,1,"Chuizaca-Espinoza, Isabel Adriana",Isabel Adriana Chuizaca-Espinoza,National Institute for Space Research (INPE)
8,8JMKD2USPTW34P/4DKBSAB,2,"Amaral, Silvana",Silvana Amaral,National Institute for Space Research (INPE)
9,8JMKD2USPTW34P/4DKBDA8,1,"Costa, Gabriel F.",Gabriel F. Costa,Federal University of Ouro Preto (UFOP)


In [ ]:
# tem que adicionar mais
MAP_INSTITUICOES = {
    "National Institute for Space Research (INPE)": "INPE",
    "Instituto Nacional de Pesquisas Espaciais": "INPE",
}

In [ ]:
df_autores_instituicoes_geoinfo["instituicao_padronizada"] = (
    df_autores_instituicoes_geoinfo["instituicao_original"]
    .map(MAP_INSTITUICOES)
    .fillna(df_autores_instituicoes_geoinfo["instituicao_original"])
)

# Países

In [ ]:
# adicionar mais conforme necessário
MAP_PAISES = {
    "br": "Brasil",
    "brazil": "Brasil",
    "brasil": "Brasil",

    "us": "Estados Unidos",
    "usa": "Estados Unidos",
    "united states": "Estados Unidos",

    "uk": "Reino Unido",
    "united kingdom": "Reino Unido"
}

In [ ]:
def padronizar_pais(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não informado"

    valor = str(valor).strip().lower()

    return MAP_PAISES.get(valor, valor.title())

In [ ]:
df_google["pais"] = (df_google["pais"].apply(padronizar_pais))
df_openalex["pais"] = (df_openalex["pais"].apply(padronizar_pais))

# Idioma

In [ ]:
# adicionar ou remover conforme necessario
MAP_IDIOMAS = {
    "en": "Inglês",
    "english": "Inglês",
    "inglês": "Inglês",

    "pt": "Português",
    "portuguese": "Português",
    "português": "Português",

    "es": "Espanhol",
    "spanish": "Espanhol",
    "espanhol": "Espanhol"
}

In [ ]:
def padronizar_idioma(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não informado"

    valor = str(valor).strip().lower()

    return MAP_IDIOMAS.get(valor, valor.title())

# Tipo documento

In [46]:
df_openalex["tipo_documento"].value_counts(dropna=False)

tipo_documento
NaN                 2950
article               29
conference-paper      13
dissertation           3
preprint               3
dataset                1
Name: count, dtype: int64

# Integração Google Scholar + OpenAlex

In [47]:
df_google["fonte_dados"] = "Google Scholar"
df_openalex["fonte_dados"] = "OpenAlex"

In [ ]:
df_citacoes = pd.concat([df_google, df_openalex], ignore_index=True, sort=False)

In [ ]:
print(f"Total de registros antes da deduplicação: {len(df_citacoes):,}")

Total de registros antes da deduplicação: 5,998


In [ ]:
contagem_fontes = (df_citacoes.groupby("id_citante")["fonte_dados"].agg(lambda x: "; ".join(sorted(set(x)))))

In [51]:
contagem_fontes.value_counts()

fonte_dados
Google Scholar; OpenAlex    2176
OpenAlex                      42
Google Scholar                42
Name: count, dtype: int64

In [ ]:
def combinar_fontes(series):
    return "; ".join(sorted(set(series.dropna().astype(str))))

In [ ]:
colunas_existentes = [c for c in COLUNAS_CONSOLIDADAS if c in df_citacoes.columns]

In [55]:
df_citacoes["id_citante"]

0       beyond the reporting of disturbed areas the us...
1       configuration assessment of deter rt a new sar...
2       revelando dinamicas da agricultura itinerante ...
3       itacart an equal area parallelogram discrete g...
4       detailed mapping of irrigated rice fields usin...
                              ...                        
5993    10 disseminacao de dados geograficos na intern...
5994    a importancia dos metadados no uso das geotecn...
5995    sistemas de informacoes geograficas elementos ...
5996    geominingvisualql uma linguagem de consulta vi...
5997                                                  NaN
Name: id_citante, Length: 5998, dtype: object

In [ ]:
def primeiro_valor(series):
    valores = series.dropna()
    valores = valores[valores.astype(str).str.strip() != ""]

    if len(valores) == 0:
        return np.nan

    return valores.iloc[0]

In [ ]:
df_final = (df_citacoes
    .groupby("id_citante", as_index=False)
    .agg({
        **{coluna: primeiro_valor for coluna in colunas_existentes},
        "fonte_dados": combinar_fontes
    })
)

In [ ]:
print(f"Registros após consolidação: {len(df_final):,}")

Registros após consolidação: 2,260


In [ ]:
display(df_final[["id_geoinfo", "id_citante", "titulo", "ano", "fonte_dados"]].head(10))

,id_geoinfo,id_citante,titulo,ano,fonte_dados
0,83LX3pFwXQZ3V9uMbiY/MdJMR,0018 2009 indexacao em bancos de dados espacia...,0018 2009 indexacao em bancos de dados espaciais,2009,Google Scholar; OpenAlex
1,83LX3pFwXQZ3V9uMbiY/MdHjv,10 disseminacao de dados geograficos na intern...,10 disseminacao de dados geograficos na internet,2005,Google Scholar; OpenAlex
2,8JMKD3MGPDW34P/487MF52,10.1002/met.70174,fine tuning lightning nowcasting for a new domain,2026,OpenAlex
3,8JMKD3MGPDW34P/4ADCN3L,10.1002/ps.6455,evidence for rice tolerance to tibraca limbati...,2021,OpenAlex
4,8JMKD3MGPDW34P/4ADCNAB,10.1007/s10661-025-14973-9,environmental indicators of forest health unde...,2026,OpenAlex
5,8JMKD3MGPDW34P/487M7M2,10.1007/s41748-026-01204-5,climate resilience beyond borders the role of ...,2026,OpenAlex
6,8JMKD3MGPDW34P/4ADCD28,10.1016/j.asoc.2026.115242,leveraging graph neural networks and mobility ...,2026,OpenAlex
7,8JMKD3MGPDW34P/4ADCMS8,10.1016/j.uclim.2024.101954,assessment of daytime and nighttime surface ur...,2024,OpenAlex
8,8JMKD3MGPDW34P/45U7K7E,10.1021/acs.est.5c05842,addressing the sustainable aviation fuel grand...,2025,OpenAlex
9,8JMKD3MGPDW34P/4ADBQ9P,10.1038/s43247-024-01542-0,unaccounted for nonforest vegetation loss in t...,2024,OpenAlex


In [ ]:
print("IDs de citantes duplicados:", df_final["id_citante"].duplicated().sum())

IDs de citantes duplicados: 0


In [ ]:
print("Relações GEOINFO → citante duplicadas:", df_final[["id_geoinfo", "id_citante"]].duplicated().sum())

Relações GEOINFO → citante duplicadas: 0


In [ ]:
inconsistencias_temporais = df_final[df_final["ano"] < df_final["ano"].astype("Int64")]

In [ ]:
mapa_ano_geoinfo = (df_geoinfo.set_index("identificador")["ano"])
df_final["ano_geoinfo"] = (df_final["id_geoinfo"].map(mapa_ano_geoinfo))

In [ ]:
inconsistencias_temporais = df_final[df_final["ano"] < df_final["ano_geoinfo"]]

print("Citações anteriores ao artigo citado:", len(inconsistencias_temporais))

Citações anteriores ao artigo citado: 24


# Resumo dos dados coletados e integrados

In [66]:
relatorio = pd.DataFrame({
    "Etapa": [
        "Artigos GEOINFO",
        "Registros Google Scholar",
        "Registros OpenAlex",
        "Registros de citações antes da integração",
        "Trabalhos citantes após consolidação"
    ],
    "Quantidade": [
        len(df_geoinfo),
        len(df_google),
        len(df_openalex),
        len(df_google) + len(df_openalex),
        len(df_final)
    ]
})

display(relatorio)

,Etapa,Quantidade
0,Artigos GEOINFO,682
1,Registros Google Scholar,2999
2,Registros OpenAlex,2999
3,Registros de citações antes da integração,5998
4,Trabalhos citantes após consolidação,2260


In [67]:
resumo_fontes = (
    df_final["fonte_dados"]
    .value_counts()
    .rename_axis("fonte")
    .reset_index(name="quantidade")
)

display(resumo_fontes)

,fonte,quantidade
0,Google Scholar; OpenAlex,2176
1,OpenAlex,42
2,Google Scholar,42


# Salvar resultados

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_geoinfo.to_csv(OUTPUT_DIR / "trabalhos_geoinfo_final.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_final.to_csv(OUTPUT_DIR / "trabalhos_citantes_final.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_autores_instituicoes_geoinfo.to_csv(OUTPUT_DIR / "autores_instituicoes_geoinfo.csv", index=False, encoding="utf-8-sig")